In [1]:
from core.model_providers.openai_llm import OpenAILLM
from core import KAGConfig

config = KAGConfig.from_yaml("configs/config_openai.yaml")
llm = OpenAILLM(config)

/vepfs-mlp2/c20250513/241404044/users/roytian/anaconda3/envs/graphweaver/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
import json

with open("data/knowledge_graph/doc2chunks.json", "r") as f:
    desc_chunks = json.load(f)

In [13]:
desc_chunks.keys()

dict_keys(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '154', '155', '156', 

In [89]:
import re


def clean_screenplay_text(content: str) -> str:
    s = content

    # 1) 修复断行导致的拆词：word-\nword  -> wordword
    #    只处理连字符紧贴在行尾的情况（最安全）
    s = re.sub(r'([A-Za-z])-\s*\n\s*([A-Za-z])', r'\1\2', s)

    # 2) 可选：修复你这个例子里已经变成 "bur- ied" 这种形式的拆词
    #    这一步更激进，建议只在你确认数据里大量存在该模式时启用
    s = re.sub(r'([A-Za-z])-\s+([a-z])', r'\1\2', s)

    # 3) 最后再压缩空白
    s = re.sub(r'\s+', ' ', s).strip()
    return s


content = desc_chunks["0"][0]["content"]
cleaned = clean_screenplay_text(content)

In [90]:
print(cleaned)

The Church of St. Peter Los Angeles. "WHOEVER YOU SEE HERE - WHATEVER YOU HEAR HERE - STAYS HERE." That's a notice on a wall. Here's another notice "NO SMOKING." Everyone is smoking. This is an AA meeting. There's a lot of Faces to look at. I don't know when we'll get to the one that's talking, but when we do it's like this. Eyes like glue. 50 years old with a face the color of a snuff-users hanky. He says this: BENNY .. after my third recovery my wife made me swear I'd never bring another bottle into the house. And I never did. I buried it under the lawn. Cut out a turf & stood it upright with a piece of tinfoil instead of a cork. So here we are out in the yard, and she's happy because I'm getting healthy in a pair of swiming shorts & no way near no booze. She decides to prune the roses. Meanwhile, I'm laying there with a straw stuck into the fucken lawn doing a quart of red .. Curious thing about drunks. Their disease often amuses them. That's how crazy I was - I was sick for half a 

In [15]:
import json
from enum import Enum
from langgraph.graph import StateGraph, END
from core.utils.format import correct_json_format
from typing import List, Dict, Any, Tuple, Set
from core.builder.manager.information_manager import InformationExtractor
import asyncio  
import os
import logging


In [16]:
from core.builder.manager.information_manager import InformationExtractor

extractor = InformationExtractor(config, llm, prompt_loader=None)

In [91]:
entity_extraction = []

for entity_type in ["event", "time_and_location", "general"]:
    existing = [f"entity name: {ent["name"]}        entity type {ent["type"]}" for ent in final_results]
    result = json.loads(extractor.extract_entities_enhanced(text=cleaned, entity_type=entity_type, extracted_entities="\n".join(existing)))
    if entity_type.lower() == "event":
        for res in result:
            res["type"] = "Event"
            res["scope"] = "local"
    entity_extraction.extend(result)

[DEBUG] messages:  [{'role': 'system', 'content': '\nYou are an excerpt in Knowledge Graph Construction tasks. \nBelow are some generals to consider during the process:\n\n1. Entity Type Identification: Do not treat category names such as Concept, Event, Object, Action as entities themselves; personal pronouns (such as "you", "I", "he") should not be used as entity names.\n2. Clear Relationship Semantics: Extracted relationships must have clear subject-object distinction, clear semantics, definite direction, and logical validity.\n3. Legal Relationship Types: Relationship type fields must strictly use the English enumeration values provided by the system, and no other forms are allowed; relationship names can be in natural language.\n4. Relationship Entity Source: Only extract relationships between already identified entities, without introducing additional unidentified entities.\n5. Extraction Decision: Relationships that cannot be clearly determined should be abandoned for extraction

In [40]:
with open("../NarrativeWeaver/core/schema/default_relation_schema.json", "r") as f:
    relation_schema = json.load(f)

In [41]:
group = relation_schema["event_role_relations"]


In [43]:
def generate_schema_text(group):
    lines = []
    for rel in group:
        if rel["direction"] == "directed":
            symbol = "-->" 
        else:
            symbol = "<-->"

        rule = "/".join(rel["from"]) + symbol + "/".join(rel["to"])
        line = f"{rel['type']}: {rel['description']}         constraint: {rule}" 
        lines.append(line)

    return "\n".join(lines)

In [44]:
def prepare_few_shot_examples(group):
    samples = []
    for rel in group:
        samples.extend(rel["samples"])

    return json.dumps(samples, ensure_ascii=False, indent=2)#.replace("{", "{{").replace("}", "}}")

In [69]:
relation_extraction_template = """
You are performing narrative relation extraction.

Goal:
Extract valid relations between already-extracted entities based strictly on the input text.
Relations MUST be grounded in the text and MUST only connect entities that are explicitly provided.

IMPORTANT CONSTRAINTS:
- You MUST NOT invent new entities.
- You MUST NOT output relations involving entities not listed in the provided entity list.
- You MUST NOT invent relation types outside the provided relation schema.
- Every extracted relation must be clearly supported by the text.

--------------------------------
INPUTS
--------------------------------

[Input Text]
{text}

[Extracted Entities]
The following entities have already been extracted.
Relations may ONLY be formed between these entities.

{entities}

[Relation Schema]
The following relation types are allowed.
Each relation type includes its definition and intended usage.

{relation_schema}

--------------------------------
RELATION EXTRACTION RULES
--------------------------------

1. Relation eligibility:
- A relation may be extracted ONLY if both the subject and object appear in the extracted entity list.
- If the text implies a relation but one side is missing from the entity list, DO NOT extract it.

2. Relation semantics:
- The relation_type MUST be exactly one of the allowed relation types.
- The relation_name MUST:
  - Be a natural-language predicate.
  - Be specific (not generic like "related_to").
  - Allow the phrase: "subject relation_name object" to read as a fluent English statement.

3. Directionality:
- Respect the semantic direction of the relation type.
- Do NOT reverse subject and object unless the schema explicitly allows symmetric relations.

4. Relation persistence (temporal stability):
You MUST assign a persistence label to each extracted relation.

- "stable": the relation is enduring and expected to remain true across the narrative unless explicitly changed
  Examples: kinship_with, affiliated_with, subordinate_to, possesses (long-term custody), long-running hostility.

- "phase": the relation holds for a meaningful period but may change later (e.g., friends become enemies, alliance breaks).
  Use when the text suggests a relationship state rather than a single action.

- "momentary": the relation is transient, tied to a single interaction, action, or immediate situation.
  Examples: pursues in one scene, threatens in a moment, briefly cooperates in a specific event.

Default policy:
- If the text evidence is limited to the current scene/action and does not imply continuation, use "momentary".
- Only use "stable" when the text explicitly signals long-term status (family, official role, membership, ongoing conflict).
- Use "phase" when the text indicates a relationship state that could plausibly evolve.

5. Description field:
- The description should briefly explain WHY this relation is extracted.
- Focus on textual evidence or narrative implication.
- Do NOT restate the relation in different words.
- May be an empty string if the relation is obvious.

6. Precision over recall:
- If a relation is ambiguous, implicit without support, or speculative, DO NOT extract it.
- Prefer missing a relation over hallucinating one.

--------------------------------
OUTPUT FORMAT
--------------------------------

Return a JSON array. MUST follow this exact structure:
[
    {{
        "subject": "Subject entity name (must match exactly)",
        "object": "Object entity name (must match exactly)",
        "relation_type": "Relation type (one of the allowed types)",
        "relation_name": "Natural-language predicate (must not be empty)",
        "persistence": "stable | phase | momentary",
        "description": "Brief extraction rationale grounded in the text"
    }}, ...
]

[Relation Few-Shot Examples]
Below are example relations demonstrating the expected output format,
relation naming style, and description rationale.

{relation_few_shot_examples}

--------------------------------
FINAL INSTRUCTIONS
--------------------------------

- Output JSON only.
- Do NOT include explanations, headings, or commentary.
- Do NOT include entities or relations not supported by the text.
- If no valid relations can be extracted, return an empty JSON array: []

"""

In [47]:
entity_type_selector = {t for rel in group for t in (rel["from"] + rel["to"])} 
entities = [f"entity_name: {ent["name"]}        type {ent["type"]}" for ent in entity_extraction if ent["type"] in entity_type_selector]
entities_text = "\n".join(entities)
relation_schema_text = generate_schema_text(group)
few_shot_samples = prepare_few_shot_examples(group)
prompt = relation_extraction_template.format(text=cleaned, entities=entities_text, relation_schema = relation_schema_text, relation_few_shot_examples=few_shot_samples)

In [70]:
import re
from typing import Dict, Iterable, Tuple

_whitespace_re = re.compile(r"\s+")

def normalize_name_basic(name: str) -> str:
    """
    Basic normalization for any entity name:
    - remove leading/trailing whitespace
    - convert any whitespace (tabs/newlines/multiple spaces) into a single space
    """
    if name is None:
        return ""
    name = str(name)
    name = _whitespace_re.sub(" ", name).strip()
    return name

def titlecase_global_name(name: str) -> str:
    """
    For global-scope names: title-case each word.
    Example: 'brandon   roy' -> 'Brandon Roy'
    Keeps punctuation reasonably (e.g., "O'Neil" -> "O'Neil" via .title()).
    """
    name = normalize_name_basic(name)
    # Python's .title() is usually fine for person/place names; if you have many edge cases, we can refine later.
    return name.title()

def canonicalize_entity_name(name: str, scope: str) -> str:
    name = normalize_name_basic(name)
    if scope == "global":
        name = titlecase_global_name(name)
    return name

def build_name_canonicalizer(entities: Iterable[Dict]) -> Tuple[Dict[str, str], Dict[str, str]]:
    """
    Returns:
    - raw2canon: map from raw name to canonical name (only for names seen in entities)
    - canon2raw: reverse map (last one wins) if you need debugging
    """
    raw2canon = {}
    canon2raw = {}
    for ent in entities:
        raw = ent.get("name", "")
        scope = ent.get("scope", "local")
        canon = canonicalize_entity_name(raw, scope)
        raw2canon[raw] = canon
        canon2raw[canon] = raw
    return raw2canon, canon2raw

def apply_canonicalization_to_entities(entities: Iterable[Dict]) -> list:
    out = []
    for ent in entities:
        ent = dict(ent)
        ent["name"] = canonicalize_entity_name(ent.get("name", ""), ent.get("scope", "local"))
        out.append(ent)
    return out

def apply_canonicalization_to_relations(relations: Iterable[Dict], raw2canon: Dict[str, str]) -> list:
    """
    Canonicalize subject/object by lookup.
    Also applies basic normalization even if not found in map.
    """
    out = []
    for rel in relations:
        rel = dict(rel)
        subj_raw = rel.get("subject", "")
        obj_raw = rel.get("object", "")
        rel["subject"] = raw2canon.get(subj_raw, normalize_name_basic(subj_raw))
        rel["object"] = raw2canon.get(obj_raw, normalize_name_basic(obj_raw))
        out.append(rel)
    return out


In [49]:
entity_extraction = apply_canonicalization_to_entities(entity_extraction)
raw2canon, _ = build_name_canonicalizer(entity_extraction)


In [51]:
messages = [{"role": "user", "content": prompt}]

result = llm.run(messages)
relation_extraction = json.loads(result[0]["content"])
relation_extraction = apply_canonicalization_to_relations(relation_extraction, raw2canon)

In [ ]:
entity_name_selector = {ent["name"] for ent in entity_extraction}
entity_name2type = {ent["name"]: ent["type"] for ent in entity_extraction}

relation_type_selector = {rel["type"] for rel in group}
rule_finder = {
    r["type"]: {
        "from": r["from"],
        "to": r["to"],
        "direction": r["direction"]
    }
    for r in group
}

feedbacks = {}

for rel in relation_extraction:
    subj = rel.get("subject", "")
    obj = rel.get("object", "")
    rtype = rel.get("relation_type", "")
    rname = rel.get("relation_name", "")

    # --- subject existence ---
    if subj not in entity_name_selector:
        feedback = (
            f"The subject [{subj}] in relation "
            f"[{subj}-({rtype})->{obj}] is not previously extracted, "
            f"consider add the entity or drop the relation"
        )
        feedbacks.setdefault("subject missing in relation", []).append(feedback)
        continue

    # --- object existence ---
    if obj not in entity_name_selector:
        feedback = (
            f"The object [{obj}] in relation "
            f"[{subj}-({rtype})->{obj}] is not previously extracted, "
            f"consider add the entity or drop the relation"
        )
        feedbacks.setdefault("object missing in relation", []).append(feedback)
        continue

    # --- relation type existence ---
    if rtype not in relation_type_selector:
        feedback = (
            f"The relation type [{rtype}] with relation name [{rname}] is undefined. "
            f"Please consider modify the relation "
            f"[{subj}-({rtype})->{obj}] or drop it."
        )
        feedbacks.setdefault("undefined relation", []).append(feedback)
        continue

    # --- entity type lookup (defensive) ---
    subject_type = entity_name2type.get(subj)
    object_type = entity_name2type.get(obj)

    if subject_type is None or object_type is None:
        feedback = (
            f"Entity type missing for relation "
            f"[{subj}-({rtype})->{obj}]. "
            f"subject_type={subject_type}, object_type={object_type}"
        )
        feedbacks.setdefault("entity type missing", []).append(feedback)
        continue

    # --- rule lookup ---
    rule = rule_finder.get(rtype)
    if rule is None:
        feedback = f"No rule defined for relation type [{rtype}]."
        feedbacks.setdefault("missing rule", []).append(feedback)
        continue

    direction = rule.get("direction")

    # --- rule satisfaction ---
    if direction == "symmetric":
        rule_satisfied = (
            (subject_type in rule["from"] and object_type in rule["to"]) or
            (subject_type in rule["to"] and object_type in rule["from"])
        )
    elif direction == "directed":
        rule_satisfied = (
            subject_type in rule["from"] and object_type in rule["to"]
        )
    else:
        feedback = (
            f"Unknown direction [{direction}] for relation type [{rtype}]."
        )
        feedbacks.setdefault("invalid direction", []).append(feedback)
        continue

    if not rule_satisfied:
        feedback = (
            f"Type constraint violated: "
            f"[{subj} ({subject_type})-({rtype})-> {obj} ({object_type})] "
            f"does not satisfy the type constraint: "
            f"{rule['from']} -> {rule['to']} for the relation type {rtype}."
        )
        feedbacks.setdefault("constraint violation", []).append(feedback)


In [57]:
feedbacks

{'constraint violation': ["Type constraint violated: [An AA speaker shares his recovery story with the group (Event)-(experiences)-> Benny (Character)] does not satisfy the type constraint: ['Character'] -> ['Event'] for the relation type experiences."]}

In [122]:
import json
from typing import Any, Dict, List, Tuple


def extract_relations_by_ground(
    entity_extraction: List[Dict[str, Any]],
    group: List[Dict[str, Any]],
    content: str,
    *,
    rid_prefix: str,
) -> Tuple[List[Dict[str, Any]], Dict[str, List[Dict[str, Any]]]]:
    """
    Extract relations for a schema group, then validate/auto-fix them with rule-based checks.

    Feedback payload format (requested):
      {"rid": str, "feedback": str, "subject": str, "object": str, "relation_type": str}

    Notes:
    - rid is GLOBAL-UNIQUE (string) using rid_prefix, e.g. "{rid_prefix}#1"
    - feedbacks is a dict keyed by error category, each value is a list of feedback payloads
    """

    # --- entity selectors ---
    entity_type_selector = {t for rel in group for t in (rel["from"] + rel["to"])}

    entities = [
        f'entity_name: {ent["name"]}        type {ent["type"]}'
        for ent in entity_extraction
        if ent["type"] in entity_type_selector
    ]
    entities_text = "\n".join(entities)

    relation_schema_text = generate_schema_text(group)
    few_shot_samples = prepare_few_shot_examples(group)

    # --- prompt ---
    prompt = relation_extraction_template.format(
        text=content,
        entities=entities_text,
        relation_schema=relation_schema_text,
        relation_few_shot_examples=few_shot_samples,
    )
    messages = [{"role": "user", "content": prompt}]

    # --- LLM call ---
    result = llm.run(messages)
    relation_extraction = json.loads(result[0]["content"])
    relation_extraction = apply_canonicalization_to_relations(relation_extraction, raw2canon)

    # --- assign GLOBAL-UNIQUE rid (per group prefix) ---
    for i, rel in enumerate(relation_extraction, start=1):
        rel["rid"] = f"{rid_prefix}#{i}"

    # --- lookup tables ---
    entity_name_selector = {ent["name"] for ent in entity_extraction}
    entity_name2type = {ent["name"]: ent["type"] for ent in entity_extraction}

    relation_type_selector = {r["type"] for r in group}
    rule_finder = {
        r["type"]: {
            "from": r["from"],
            "to": r["to"],
            "direction": r["direction"],
        }
        for r in group
    }

    # feedbacks[key] = list[{rid, feedback, subject, object, relation_type}]
    feedbacks: Dict[str, List[Dict[str, Any]]] = {}

    def add_feedback(key: str, rid: str, feedback: str, subj: str, obj: str, rtype: str) -> None:
        payload = {
            "rid": rid,
            "feedback": feedback,
            "subject": subj,
            "object": obj,
            "relation_type": rtype,
        }
        feedbacks.setdefault(key, []).append(payload)

    # --- validation & auto-fix ---
    for rel in relation_extraction:
        rid = rel.get("rid", "")
        subj = rel.get("subject", "")
        obj = rel.get("object", "")
        rtype = rel.get("relation_type", "")
        rname = rel.get("relation_name", "")

        # --- subject existence ---
        if subj not in entity_name_selector:
            fb = (
                f"Subject [{subj}] is not previously extracted. "
                f"Consider adding the entity or dropping the relation."
            )
            add_feedback("subject error", rid, fb, subj, obj, rtype)
            rel["conf"] = 0.0
            continue

        # --- object existence ---
        if obj not in entity_name_selector:
            fb = (
                f"Object [{obj}] is not previously extracted. "
                f"Consider adding the entity or dropping the relation."
            )
            add_feedback("object error", rid, fb, subj, obj, rtype)
            rel["conf"] = 0.0
            continue

        # --- relation type existence ---
        if rtype not in relation_type_selector:
            fb = (
                f"Relation type [{rtype}] (name [{rname}]) is not defined in schema. "
                f"Consider modifying or dropping the relation."
            )
            add_feedback("undefined relation error", rid, fb, subj, obj, rtype)
            rel["conf"] = 0.0
            continue

        # --- subject type lookup ---
        subject_type = entity_name2type.get(subj)
        if subject_type is None:
            fb = (
                f"Subject [{subj}] exists but its type is missing in entity_name2type. "
                f"Fix entity typing or drop the relation."
            )
            add_feedback("subject error", rid, fb, subj, obj, rtype)
            rel["conf"] = 0.0
            continue

        # --- object type lookup ---
        object_type = entity_name2type.get(obj)
        if object_type is None:
            fb = (
                f"Object [{obj}] exists but its type is missing in entity_name2type. "
                f"Fix entity typing or drop the relation."
            )
            add_feedback("object error", rid, fb, subj, obj, rtype)
            rel["conf"] = 0.0
            continue

        # --- rule lookup ---
        rule = rule_finder.get(rtype)
        if rule is None:
            fb = f"No rule defined for relation type [{rtype}]."
            add_feedback("missing rule error", rid, fb, subj, obj, rtype)
            rel["conf"] = 0.0
            continue

        direction = rule.get("direction")

        # --- rule satisfaction ---
        if direction == "symmetric":
            rule_satisfied = (
                (subject_type in rule["from"] and object_type in rule["to"])
                or (subject_type in rule["to"] and object_type in rule["from"])
            )
        elif direction == "directed":
            rule_satisfied = (subject_type in rule["from"] and object_type in rule["to"])
        else:
            fb = f"Unknown direction [{direction}] for relation type [{rtype}]."
            add_feedback("invalid direction error", rid, fb, subj, obj, rtype)
            rel["conf"] = 0.0
            continue

        if not rule_satisfied:
            # AUTO-FIX: flip if it would satisfy the typed constraint
            if direction == "directed":
                flipped_satisfied = (object_type in rule["from"] and subject_type in rule["to"])
                if flipped_satisfied:
                    rel["subject"], rel["object"] = obj, subj
                    rel["auto_fixed"] = True
                    rel["fix_reason"] = "swap_subject_object_to_satisfy_type_constraint"
                    rel["conf"] = 0.5
                    continue

            fb = (
                f"Type constraint violated: subject type [{subject_type}] and object type [{object_type}] "
                f"do not satisfy {rule['from']} -> {rule['to']}."
            )
            add_feedback("constraint violation error", rid, fb, subj, obj, rtype)
            rel["conf"] = 0.0
        else:
            rel["conf"] = 0.8

    return relation_extraction, feedbacks


In [123]:
from tqdm import tqdm

all_relations = {}   # rid -> relation dict
all_feedbacks = {}   # error_key -> list[feedback dict]

for relation_group in tqdm(relation_schema):
    group = relation_schema[relation_group]

    relation_extraction, feedbacks = extract_relations_by_ground(
        entity_extraction=entity_extraction,
        group=group,
        content=cleaned,
        rid_prefix=str(relation_group),   # 关键：group 名作为 prefix
    )

    # 1) 累积关系：dict by rid
    for rel in (relation_extraction or []):
        rid = rel.get("rid")
        if not rid:
            raise ValueError(f"Missing rid in relation: {rel}")
        if rid in all_relations:
            raise ValueError(f"Duplicate rid detected: {rid}")
        all_relations[rid] = rel

    # 2) 累积反馈：按 key 合并（rid 已经天然一致）
    if feedbacks:
        for k, items in feedbacks.items():
            all_feedbacks.setdefault(k, [])
            if isinstance(items, list):
                all_feedbacks[k].extend(items)
            else:
                all_feedbacks[k].append(items)

# 3) 一致性校验：确保每条 feedback 的 rid 都能定位到关系
missing_rids = []
for k, items in all_feedbacks.items():
    for fb in items:
        rid = fb.get("rid")
        if rid not in all_relations:
            missing_rids.append((k, rid, fb.get("message", "")))

if missing_rids:
    # 打印前几个帮助 debug
    preview = "\n".join([f"{k}: {rid} -> {msg}" for k, rid, msg in missing_rids[:10]])
    raise ValueError(
        f"Found feedbacks pointing to unknown rids. Count={len(missing_rids)}\n{preview}"
    )


100%|██████████| 6/6 [00:25<00:00,  4.33s/it]


In [125]:
all_feedbacks["subject error"][0]

{'rid': 'event_role_relations#1',
 'feedback': 'Subject [Benny] is not previously extracted. Consider adding the entity or dropping the relation.',
 'subject': 'Benny',
 'object': 'Benny shares his recovery story at an Alcoholics Anonymous meeting',
 'relation_type': 'performs'}

In [131]:
relation_type_selected = all_feedbacks["subject error"][0]["relation_type"]

In [149]:
feedback = all_feedbacks["subject error"][0]["feedback"]

feedback

'Subject [Benny] is not previously extracted. Consider adding the entity or dropping the relation.'

In [134]:
global_rule_finder = {}

for group in relation_schema.values():   # 如果 relation_schema 是 dict
    for r in group:
        rtype = r["type"]
        if rtype in global_rule_finder:
            raise ValueError(
                f"Duplicate relation_type detected across groups: {rtype}"
            )
        global_rule_finder[rtype] = {
            "from": r["from"],
            "to": r["to"],
            "direction": r["direction"],
        }


In [138]:
global_rule_finder[relation_type_selected] # 如果没有就drop; subject就是from, object就是to

{'from': ['Character', 'Object', 'Concept'],
 'to': ['Event'],
 'direction': 'directed'}

In [145]:
candidate_list = global_rule_finder[relation_type_selected]["from"] # 如果没有就drop; subject就是from, object就是to

candidate_list

['Character', 'Object', 'Concept']

In [139]:
with open("../NarrativeWeaver/core/schema/default_entity_schema.json", "r") as f:
    entity_schema = json.load(f)

In [142]:
entity_type_description = {ent["type"]: ent["description"] for ent in entity_schema}

In [151]:
entity_type_description_text = "\n".join([f"{ent}: {entity_type_description[ent]}" for ent in candidate_list])

In [153]:
print(entity_type_description_text)

Character: A concrete individual entity that participates in the narrative (human or anthropomorphized agent).
Object: A concrete physical item that can be held, used, hidden, damaged, or exchanged, and is narratively relevant.
Concept: An abstract, collective, or institutional entity such as an organization, faction, ideology, system, or social group.


In [ ]:
entity_error_fix_prompt = """
You are fixing a relation extraction error in a narrative knowledge graph.

TASK
You are given:
1) The original scene text.
2) A set of candidate entity types that the missing entity is allowed to be, based on the relation type constraint.
3) Short descriptions for each candidate type.
4) A feedback item that indicates what went wrong (e.g., subject missing / object missing).

Your goal is to decide whether to ADD a missing entity (only if it is explicitly grounded in the text)
or DROP the relation (if the missing entity cannot be supported).

CRITICAL CONSTRAINTS
- Do NOT invent information that is not in the text.
- If the missing entity is not explicitly mentioned (or cannot be uniquely identified) in the text, you MUST choose "drop".
- If you choose "add", the entity MUST:
  - Use an entity type from Candidate Entity Types.
  - Have a clean canonical name (trim spaces, single spaces between words, no line breaks).
  - Be grounded in the text (the description should reference what is stated, not guesses).
  - Include a scope: "global" if it is a recurring named entity likely to appear across scenes (e.g., "Benny", "AA", "Los Angeles"),
    otherwise "local" if it is only a one-off mention in this scene.

INPUTS
[Scene Text]
{content}

[Candidate Entity Types]
{candidate_entity_types}

[Candidate Type Descriptions]
{candidate_entity_descriptions}

[Feedback]
{feedback}

DECISION RULES
Choose exactly ONE of the following:
1) "add":
   - Only if you can point to an explicit mention in the Scene Text that corresponds to the missing entity.
   - The entity name should match the surface form in the text (after simple cleanup).
2) "drop":
   - If the missing entity is not clearly present in the Scene Text,
     OR there are multiple plausible entities and you cannot disambiguate,
     OR adding it would require assumptions.

OUTPUT FORMAT
Return ONLY one JSON object, with no extra text.

If decision is "add":
{{
  "decision": "add",
  "output": {{
    "name": "<string>",
    "description": "<string grounded in the text>",
    "type": "<one of Candidate Entity Types>",
    "scope": "global" | "local"
  }}
}}

If decision is "drop":
{{
  "decision": "drop",
  "output": {{}}
}}
"""

In [157]:
prompt = entity_error_fix_prompt.format(content=cleaned, feedback=feedback, candidate_entity_types=candidate_list, candidate_entity_descriptions=entity_type_description_text)

In [158]:
def build_simple_message(prompt):
    return [{"role": "user", "content": prompt}]

In [159]:
llm.run(build_simple_message(prompt))

[Message({'role': 'assistant', 'content': '{\n  "decision": "add",\n  "output": {\n    "name": "Benny",\n    "description": "A participant in the AA meeting who shares his story about burying a bottle of alcohol under the lawn and recovering from alcoholism.",\n    "type": "Character",\n    "scope": "global"\n  }\n}'})]

In [162]:
all_feedbacks["constraint violation error"][0]

{'rid': 'spatiotemporal_relations#8',
 'feedback': "Type constraint violated: subject type [Character] and object type [Concept] do not satisfy ['Character', 'Object', 'Concept'] -> ['TimePoint'].",
 'subject': 'John Berlin',
 'object': 'AA meeting',
 'relation_type': 'present_on'}

In [167]:
from collections import defaultdict
from typing import Dict, Set, Tuple, Any

def build_typepair_relation_index(
    global_rule_finder: Dict[str, Dict[str, Any]]
) -> Dict[Tuple[str, str], Set[str]]:
    """
    Build a simplified index:
      (type_a, type_b) -> {relation_type, ...}

    - Ignores direction
    - Treats (A, B) and (B, A) as the same key
    - Deduplicates relation types
    """
    index = defaultdict(set)

    for relation_type, rule in global_rule_finder.items():
        from_types = rule.get("from", [])
        to_types = rule.get("to", [])

        # 所有可能的类型组合（无向）
        for t1 in from_types:
            for t2 in to_types:
                key = tuple(sorted((t1, t2)))
                index[key].add(relation_type)

    return dict(index)

def get_allowed_relations_between_types(
    typepair_index: Dict[Tuple[str, str], List[Dict[str, Any]]],
    subject_type: str,
    object_type: str,
) -> List[Dict[str, Any]]:
    """
    Return allowed relations if subject_type -> object_type is valid.
    """
    return typepair_index.get((subject_type, object_type), [])


In [168]:
typepair_index = build_typepair_relation_index(global_rule_finder)

allowed = get_allowed_relations_between_types(typepair_index, "Character", "Concept")
allowed

{'affiliated_with', 'affinity_with', 'hostility_with', 'is_a'}

In [95]:
from collections import defaultdict
import json

def _rel_brief(rel: dict) -> str:
    rtype = rel.get("relation_type", "")
    rname = rel.get("relation_name", "")
    desc = rel.get("description", "")
    fixed = rel.get("auto_fixed", False)
    fix_reason = rel.get("fix_reason", "")
    extra = ""
    if fixed:
        extra = f"  [auto_fixed: {fix_reason}]"
    if desc:
        return f"- {rtype} | {rname}{extra}\n  desc: {desc}"
    return f"- {rtype} | {rname}{extra}"

def find_multi_relations(relations, *, undirected: bool = False, min_count: int = 2):
    """
    Returns a dict:
      key -> list[relation]
    key is either:
      (subject, object) if undirected=False
      (min(subject, object), max(subject, object)) if undirected=True
    """
    buckets = defaultdict(list)
    for rel in relations:
        s = rel.get("subject", "")
        o = rel.get("object", "")
        if not s or not o:
            continue

        key = (s, o)
        if undirected:
            key = tuple(sorted([s, o]))

        buckets[key].append(rel)

    # filter only those with multiple relations
    multi = {k: v for k, v in buckets.items() if len(v) >= min_count}
    return multi

def print_multi_relations(multi: dict, *, title: str = "", max_pairs: int | None = None):
    if title:
        print(title)
        print("=" * len(title))

    items = list(multi.items())
    # sort by number of relations desc
    items.sort(key=lambda kv: len(kv[1]), reverse=True)

    if max_pairs is not None:
        items = items[:max_pairs]

    for (a, b), rels in items:
        print(f"\n[{a}]  <->  [{b}]   (count={len(rels)})")
        for rel in rels:
            print(_rel_brief(rel))

# 1) 有向：同一 subject->object 多关系
multi_directed = find_multi_relations(all_relations, undirected=False, min_count=2)
print_multi_relations(multi_directed, title="Multiple relations between the same directed pair (subject -> object)")



Multiple relations between the same directed pair (subject -> object)

[John Berlin]  <->  [AA meeting]   (count=2)
- affiliated_with | attends
  desc: John Berlin is present at the AA meeting, as indicated by the narrative describing him after the meeting activity and sharing session.
- present_on | attends
  desc: John Berlin is present and about to speak at the AA meeting, indicating his participation in the event at that time.

[straw]  <->  [the yard]   (count=2)
- part_of | is inserted into
  desc: The narrator describes using a straw stuck into the lawn to drink alcohol, indicating a temporary physical placement of the straw in the yard.
- located_at | is inserted into
  desc: The straw is described as being stuck into the lawn in the yard during Benny's story, indicating a temporary physical placement.


In [96]:
all_relations

[{'subject': 'Benny',
  'object': 'Benny shares his recovery story at an Alcoholics Anonymous meeting',
  'relation_type': 'performs',
  'relation_name': 'shares his story in',
  'persistence': 'momentary',
  'description': 'The text describes Benny speaking and recounting his personal history during the AA meeting, indicating he is the one performing the act of sharing.',
  'rid': 1},
 {'subject': 'John Berlin',
  'object': 'John Berlin prepares to speak at the AA meeting',
  'relation_type': 'performs',
  'relation_name': 'prepares to speak in',
  'persistence': 'momentary',
  'description': 'The narrative introduces John Berlin immediately after indicating someone is about to speak, linking him directly to the preparation to share at the meeting.',
  'rid': 2},
 {'subject': 'AA meeting',
  'object': 'Benny shares his recovery story at an Alcoholics Anonymous meeting',
  'relation_type': 'undergoes',
  'relation_name': 'hosts',
  'persistence': 'momentary',
  'description': 'The even